In [1]:
# -*- coding: utf-8 -*-
"""1m + 5m rank fusion inference file for BigAlpha.

这个文件是单文件自包含融合推理：只需要同目录下有融合版 model.json。
model.json 内部包含 1m 子模型和 5m 子模型，推理时分别读取官方
bigalpha_2026_stock_bar1m / bigalpha_2026_stock_bar5m，然后做每日 rank 加权融合。
"""

from __future__ import annotations

import glob
import json
import math
import os
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import dai
except ModuleNotFoundError:
    try:
        from bigquant import dai
    except ModuleNotFoundError:
        dai = None


ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = ROOT.parent


def _find_model_path() -> Path:
    """兼容 .py 与 .ipynb 平台环境，自动查找融合版 model.json。

    注意：平台工作目录里可能残留单模型 model.json。这里必须检查
    fusion_type，避免误加载 1m/5m 单模型导致没有输出分数。
    """
    def is_fusion_model(path: Path) -> bool:
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            return False
        return payload.get("fusion_type") == "daily_rank_weighted_sum"

    env_path = os.environ.get("DIFFMLANET_MODEL_PATH")
    if env_path and Path(env_path).exists():
        path = Path(env_path)
        if not is_fusion_model(path):
            raise RuntimeError(f"DIFFMLANET_MODEL_PATH 指向的不是融合模型: {path}")
        return path

    candidates: list[Path] = [
        ROOT / "model.json",
        Path.cwd() / "model.json",
        Path("/home/aiuser/work") / "model.json",
    ]
    try:
        candidates.extend(Path.cwd().glob("**/model.json"))
    except Exception:
        pass

    seen = set()
    unique_candidates: list[Path] = []
    for path in candidates:
        key = str(path)
        if key not in seen:
            seen.add(key)
            unique_candidates.append(path)

    existing = [path for path in unique_candidates if path.exists()]
    for path in existing:
        if is_fusion_model(path):
            return path

    checked = "\n".join(str(path) for path in existing[:30])
    if checked:
        raise FileNotFoundError(f"找不到融合版 model.json；以下 model.json 都不是 fusion 模型：\n{checked}")
    checked = "\n".join(str(path) for path in unique_candidates[:30])
    raise FileNotFoundError(f"找不到融合版 model.json，已检查：\n{checked}")


MODEL_PATH = _find_model_path()

SEQ_LEN = 64
BATCH = int(os.environ.get("DIFFMLANET_BATCH", "512"))
LOCAL_INFER_DATA_PATH = os.environ.get(
    "LOCAL_INFER_DATA_PATH",
    str(PROJECT_ROOT / "local_data/e2e_bar1m_validation_2024"),  # fusion local helper keeps 1m default only
)
LOCAL_SCORE_PATH = os.environ.get(
    "DIFFMLANET_LOCAL_SCORE_PATH",
    str(PROJECT_ROOT / "outputs/re_dm_fusion_score_data_local.csv"),
)
# 融合版需要分别跑 1m/5m 两套模型。默认分块比单模型大一些，
# 减少 dai.query 次数，避免平台公榜推理超时；如显存/内存不够可设小。
INFER_CHUNK = int(os.environ.get("DIFFMLANET_INFER_CHUNK", "80"))
MIN_DAILY_COVERAGE = float(os.environ.get("DIFFMLANET_MIN_DAILY_COVERAGE", "0.60"))
SCORE_POSTPROCESS = os.environ.get("DIFFMLANET_SCORE_POSTPROCESS", "daily_rank").lower()

OHLC_COLS = ["high", "open", "low", "close"]
ASK_PRICE_COLS = [f"ask_price{i}" for i in range(1, 4)]
BID_PRICE_COLS = [f"bid_price{i}" for i in range(1, 4)]
ASK_VOLUME_COLS = [f"ask_volume{i}" for i in range(1, 4)]
BID_VOLUME_COLS = [f"bid_volume{i}" for i in range(1, 4)]
ASK_ORDER_COLS = [f"ask_num_orders{i}" for i in range(1, 4)]
BID_ORDER_COLS = [f"bid_num_orders{i}" for i in range(1, 4)]
PRICE_COLS = OHLC_COLS + ASK_PRICE_COLS + BID_PRICE_COLS
LOG1P_COLS = (
    ["deal_number", "volume", "amount"]
    + ASK_VOLUME_COLS
    + BID_VOLUME_COLS
    + ASK_ORDER_COLS
    + BID_ORDER_COLS
)
FEATURE_COLS = (
    ["adjust_factor"]
    + OHLC_COLS
    + ["deal_number", "volume", "amount"]
    + ASK_PRICE_COLS
    + BID_PRICE_COLS
    + ASK_VOLUME_COLS
    + BID_VOLUME_COLS
    + ASK_ORDER_COLS
    + BID_ORDER_COLS
)
FEATURE_SCHEMA_VERSION = "fusion_rank_1m_5m_v1"
SCHEMA_BY_FREQ = {
    "1m": "e2e_bar1m_full_3level_v1",
    "5m": "e2e_bar5m_full_3level_v1",
}
TABLE_BY_FREQ = {
    "1m": "bigalpha_2026_stock_bar1m",
    "5m": "bigalpha_2026_stock_bar5m",
}


def log(message: str, **kwargs: Any) -> None:
    extra = " ".join(f"{key}={value}" for key, value in kwargs.items())
    print(f"[reDMFusion][infer] {message}" + (f" {extra}" if extra else ""))


def full_day_bounds(start_date: str, end_date: str) -> tuple[str, str]:
    start = pd.Timestamp(start_date).normalize()
    end = pd.Timestamp(end_date).normalize() + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    return start.strftime("%Y-%m-%d %H:%M:%S"), end.strftime("%Y-%m-%d %H:%M:%S")


def to_canonical(df: pd.DataFrame, *, is_local: bool) -> pd.DataFrame:
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    if is_local:
        # 本地 parquet 中价格类字段可能用 -1 表示缺失。
        # 需要先把所有价格字段的 -1 置空，再做 /100 缩放，避免 -1 被错误变成 -0.01。
        for col in PRICE_COLS:
            if col in df.columns:
                df.loc[df[col] == -1, col] = np.nan
        for col in PRICE_COLS + ["amount"]:
            if col in df.columns:
                df[col] = df[col].astype("float64") / 100.0
        if "instrument" not in df.columns:
            df["instrument"] = df["instrument_id"].astype(str)
    else:
        df["instrument"] = df["instrument"].astype(str)
    missing = [col for col in FEATURE_COLS if col not in df.columns]
    if missing:
        raise ValueError(f"输入数据缺少模型字段: {missing}")
    return (
        df[["date", "instrument"] + FEATURE_COLS]
        .dropna(subset=["date", "instrument"])
        .sort_values(["instrument", "date"])
        .reset_index(drop=True)
    )


def _prepare_one_day(day_df: pd.DataFrame) -> tuple[np.ndarray, float] | None:
    day_df = day_df.sort_values("date")
    if len(day_df) < SEQ_LEN:
        return None
    feature_df = day_df[FEATURE_COLS].copy()
    for col in LOG1P_COLS:
        feature_df[col] = np.log1p(pd.to_numeric(feature_df[col], errors="coerce").clip(lower=0))
    feature_df = feature_df.replace([np.inf, -np.inf], np.nan).ffill().fillna(0.0)
    window = feature_df.iloc[-SEQ_LEN:].to_numpy(dtype=np.float32)
    close = pd.to_numeric(day_df["close"], errors="coerce").replace([np.inf, -np.inf], np.nan).ffill()
    close_value = float(close.iloc[-1]) if len(close) else np.nan
    return window, close_value


def build_samples(
    df: pd.DataFrame,
    *,
    start_date: str,
    end_date: str,
    mode: str,
    instruments: list[str] | None = None,
    stats: tuple[np.ndarray, np.ndarray] | None = None,
    normalize: bool = True,
) -> tuple[np.ndarray, np.ndarray | None, pd.DataFrame, tuple[np.ndarray, np.ndarray] | None]:
    start_ts = pd.Timestamp(start_date)
    end_ts = pd.Timestamp(end_date)
    data = df[(df["date"] >= start_ts) & (df["date"] <= end_ts)].copy()
    if instruments is not None:
        wanted = {str(item) for item in instruments}
        data = data[data["instrument"].astype(str).isin(wanted)]
    if data.empty:
        raise RuntimeError(f"指定区间没有数据: {start_date} ~ {end_date}")

    windows: list[np.ndarray] = []
    labels: list[np.float32] = []
    keys: list[tuple[pd.Timestamp, str]] = []
    for instrument, instrument_df in data.groupby("instrument", sort=False):
        daily: list[tuple[pd.Timestamp, np.ndarray, float]] = []
        for day, day_df in instrument_df.groupby(instrument_df["date"].dt.normalize(), sort=True):
            prepared = _prepare_one_day(day_df)
            if prepared is not None:
                window, close = prepared
                daily.append((pd.Timestamp(day), window, close))
        for index, (day, window, close) in enumerate(daily):
            label = None
            if index + 1 < len(daily):
                next_close = daily[index + 1][2]
                if np.isfinite(close) and np.isfinite(next_close) and close > 0:
                    value = next_close / close - 1.0
                    if np.isfinite(value):
                        label = np.float32(value)
            if mode == "train" and label is None:
                continue
            windows.append(window)
            keys.append((day, str(instrument)))
            if mode == "train":
                labels.append(label)

    if not windows:
        raise RuntimeError(f"没有构造出样本: mode={mode}, {start_date} ~ {end_date}")
    X = np.stack(windows).astype(np.float32)
    if normalize:
        if stats is None:
            raise ValueError("推理或标准化时必须提供训练集 stats")
        mean, std = stats
        X = ((X - mean) / std).astype(np.float32)
    index_df = pd.DataFrame(keys, columns=["date", "instrument"])
    y = np.asarray(labels, dtype=np.float32) if mode == "train" else None
    return X, y, index_df, stats



@dataclass
class DiffMLANetConfig:
    n_feat: int = 26
    seq_len: int = 64
    d_model: int = 128
    conv_windows: list[int] = field(default_factory=lambda: [4, 8, 16, 32, 64])
    conv_channels: int = 96
    dropout: float = 0.2
    aeccm_inplanes: int = 8
    aeccm_ratio: float = 0.5
    aeccm_low_rank_ratio: float = 0.25
    diffusion_enabled: bool = True
    diffusion_steps: int = 128
    diffusion_beta_start: float = 1e-4
    diffusion_beta_end: float = 0.02
    diffusion_hidden_dim: int = 64
    diffusion_time_dim: int = 64
    diffusion_num_blocks: int = 2
    diffusion_inference_step: int = 64
    diffusion_denoise_eta: float = 0.2
    lambda_diff: float = 0.01


class RevIN(nn.Module):
    def __init__(self, num_variates: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(1, num_variates, 1))
        self.beta = nn.Parameter(torch.zeros(1, num_variates, 1))

    def forward(self, x: torch.Tensor):
        x_feature = x.permute(0, 2, 1)
        mean = x_feature.mean(dim=-1, keepdim=True)
        var = x_feature.var(dim=-1, unbiased=False, keepdim=True)
        normalized = (x_feature - mean) * var.clamp(min=self.eps).rsqrt()
        scaled = normalized * self.gamma + self.beta
        return scaled.permute(0, 2, 1)


class SinusoidalTimeEmbedding(nn.Module):
    """re-mLANet4 diffusion time embedding, inlined for two-file submission."""

    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, timesteps: torch.Tensor) -> torch.Tensor:
        device = timesteps.device
        half_dim = self.dim // 2
        if half_dim == 0:
            return timesteps.float().unsqueeze(-1)
        scale = math.log(10000) / max(half_dim - 1, 1)
        freqs = torch.exp(torch.arange(half_dim, device=device) * -scale)
        args = timesteps.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([args.sin(), args.cos()], dim=1)
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0, 1))
        return emb


def _group_count(channels: int, max_groups: int = 8) -> int:
    for groups in range(min(max_groups, channels), 0, -1):
        if channels % groups == 0:
            return groups
    return 1


class ResidualConvBlock(nn.Module):
    """Residual 1D conv block used by the re-mLANet4 diffusion denoiser."""

    def __init__(self, channels: int, time_dim: int, dilation: int = 1):
        super().__init__()
        padding = dilation
        groups = _group_count(channels)
        self.norm1 = nn.GroupNorm(groups, channels)
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, padding=padding, dilation=dilation)
        self.time_proj = nn.Linear(time_dim, channels)
        self.norm2 = nn.GroupNorm(groups, channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, padding=padding, dilation=dilation)
        self.act = nn.SiLU()

    def forward(self, x: torch.Tensor, time_emb: torch.Tensor) -> torch.Tensor:
        residual = x
        x = self.conv1(self.act(self.norm1(x)))
        x = x + self.time_proj(time_emb).unsqueeze(-1)
        x = self.conv2(self.act(self.norm2(x)))
        return x + residual


class DiffusionDenoiser(nn.Module):
    """Lightweight DDPM noise predictor copied from re-mLANet4 and kept self-contained."""

    def __init__(
        self,
        seq_len: int,
        enc_in: int,
        num_steps: int = 128,
        beta_start: float = 1e-4,
        beta_end: float = 0.02,
        hidden_dim: int = 64,
        time_dim: int = 64,
        num_blocks: int = 2,
        inference_step: int | None = None,
    ):
        super().__init__()
        self.seq_len = seq_len
        self.enc_in = enc_in
        self.num_steps = num_steps
        self.inference_step = num_steps // 2 if inference_step is None else inference_step

        betas = torch.linspace(beta_start, beta_end, num_steps, dtype=torch.float32)
        alphas = 1.0 - betas
        alpha_bars = torch.cumprod(alphas, dim=0)
        self.register_buffer("betas", betas, persistent=False)
        self.register_buffer("alphas", alphas, persistent=False)
        self.register_buffer("alpha_bars", alpha_bars, persistent=False)

        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )
        self.input_proj = nn.Conv1d(1, hidden_dim, kernel_size=3, padding=1)
        self.blocks = nn.ModuleList(
            [ResidualConvBlock(hidden_dim, time_dim, dilation=2 ** (idx % 3)) for idx in range(num_blocks)]
        )
        self.output_norm = nn.GroupNorm(_group_count(hidden_dim), hidden_dim)
        self.output_proj = nn.Conv1d(hidden_dim, 1, kernel_size=3, padding=1)
        self.act = nn.SiLU()

    def _extract(self, values: torch.Tensor, timesteps: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        gathered = values.gather(0, timesteps).to(device=x.device, dtype=x.dtype)
        return gathered.view(-1, 1, 1)

    def _expand_timesteps(self, timesteps: torch.Tensor, channels: int) -> torch.Tensor:
        return timesteps.unsqueeze(1).expand(-1, channels).reshape(-1)

    def q_sample(
        self,
        x0: torch.Tensor,
        timesteps: torch.Tensor,
        noise: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_alpha_bar = self._extract(self.alpha_bars.sqrt(), timesteps, x0)
        sqrt_one_minus_alpha_bar = self._extract((1.0 - self.alpha_bars).sqrt(), timesteps, x0)
        return sqrt_alpha_bar * x0 + sqrt_one_minus_alpha_bar * noise, noise

    def predict_noise(self, x_t: torch.Tensor, timesteps: torch.Tensor) -> torch.Tensor:
        batch, length, channels = x_t.shape
        x = x_t.permute(0, 2, 1).reshape(batch * channels, 1, length)
        time_emb = self.time_mlp(self._expand_timesteps(timesteps, channels))

        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x, time_emb)
        x = self.output_proj(self.act(self.output_norm(x)))
        return x.reshape(batch, channels, length).permute(0, 2, 1)

    def predict_clean_from_noise(
        self,
        x_t: torch.Tensor,
        timesteps: torch.Tensor,
        pred_noise: torch.Tensor,
    ) -> torch.Tensor:
        sqrt_alpha_bar = self._extract(self.alpha_bars.sqrt(), timesteps, x_t)
        sqrt_one_minus_alpha_bar = self._extract((1.0 - self.alpha_bars).sqrt(), timesteps, x_t)
        return (x_t - sqrt_one_minus_alpha_bar * pred_noise) / sqrt_alpha_bar.clamp(min=1e-8)

    def training_loss(
        self,
        x0: torch.Tensor,
        timesteps: torch.Tensor | None = None,
        noise: torch.Tensor | None = None,
    ) -> dict[str, torch.Tensor]:
        batch = x0.shape[0]
        if timesteps is None:
            timesteps = torch.randint(0, self.num_steps, (batch,), device=x0.device)
        x_t, noise = self.q_sample(x0, timesteps, noise)
        pred_noise = self.predict_noise(x_t, timesteps)
        loss = F.mse_loss(pred_noise, noise)
        x_clean = self.predict_clean_from_noise(x_t, timesteps, pred_noise)
        return {
            "loss": loss,
            "x_clean": x_clean,
            "x_t": x_t,
            "pred_noise": pred_noise,
            "noise": noise,
            "timesteps": timesteps,
        }

    def denoise(self, x_t: torch.Tensor, timesteps: torch.Tensor | None = None) -> torch.Tensor:
        batch = x_t.shape[0]
        if timesteps is None:
            step = max(0, min(int(self.inference_step), self.num_steps - 1))
            timesteps = torch.full((batch,), step, device=x_t.device, dtype=torch.long)
        pred_noise = self.predict_noise(x_t, timesteps)
        return self.predict_clean_from_noise(x_t, timesteps, pred_noise)

    def forward(
        self,
        x: torch.Tensor,
        return_loss: bool = False,
        timesteps: torch.Tensor | None = None,
    ) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:
        if return_loss:
            diffusion = self.training_loss(x, timesteps=timesteps)
            return diffusion["x_clean"], diffusion["loss"]
        return self.denoise(x, timesteps=timesteps)


def split_window_chains(windows: list[int]) -> list[list[int]]:
    """Same divisibility-chain idea as re-mLANet4/layers/PyramidalRNNEmbedding.py."""
    windows = sorted(int(item) for item in windows)
    used = np.zeros(len(windows), dtype=np.int8)
    chains: list[list[int]] = []
    while used.sum() < len(windows):
        chain = []
        last = None
        for index in range(len(windows) - 1, -1, -1):
            value = windows[index]
            if used[index] == 0 or index == 0:
                if last is None:
                    chain.append(value)
                    used[index] = 1
                    last = value
                elif last % value == 0:
                    chain.append(value)
                    used[index] = 1
                    last = value
        if chain:
            chains.append(sorted(chain))
    return chains


def count_chain_windows(chains: list[list[int]]) -> int:
    return sum(len(chain) for chain in chains)


class ConvRNNBlock(nn.Module):
    def __init__(self, kernel: int, conv_channels: int, hidden_size: int):
        super().__init__()
        self.conv = nn.Conv1d(1 if kernel == 0 else conv_channels, conv_channels, kernel_size=max(1, kernel), stride=max(1, kernel))
        self.gru = nn.GRU(conv_channels, hidden_size, batch_first=True)

    def forward_conv(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(x)

    def forward_rnn(self, x: torch.Tensor) -> torch.Tensor:
        _, hidden = self.gru(x.permute(0, 2, 1))
        return hidden[-1]


class PyramidalRNNEmbedding(nn.Module):
    """re-mLANet4 PRE: pyramidal convolution chains with GRU summaries."""

    def __init__(self, config: DiffMLANetConfig):
        super().__init__()
        self.config = config
        self.chains = split_window_chains(config.conv_windows)
        window_count = count_chain_windows(self.chains)
        self.rate = nn.Parameter(torch.ones(window_count, 1) / max(window_count, 1))
        self.temperature = 0.002

        counts = {window: 0 for window in config.conv_windows}
        for chain in self.chains:
            for window in chain:
                counts[window] += 1
        base_hidden = max(1, config.d_model // len(config.conv_windows))
        self.hidden_by_window = {
            window: max(1, base_hidden // max(counts[window], 1))
            for window in config.conv_windows
        }
        out_dim = sum(self.hidden_by_window[window] for chain in self.chains for window in chain)

        modules = []
        for chain in self.chains:
            chain_blocks = []
            previous_window = 1
            for position, window in enumerate(chain):
                kernel = int(window / previous_window)
                hidden = self.hidden_by_window[window]
                block = nn.ModuleDict(
                    {
                        "conv": nn.Conv1d(
                            1 if position == 0 else config.conv_channels,
                            config.conv_channels,
                            kernel_size=max(1, kernel),
                            stride=max(1, kernel),
                        ),
                        "gru": nn.GRU(config.conv_channels, hidden, batch_first=True),
                    }
                )
                chain_blocks.append(block)
                previous_window = window
            modules.append(nn.ModuleList(chain_blocks))
        self.chain_modules = nn.ModuleList(modules)
        self.out = nn.Linear(out_dim, config.d_model)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, seq_len, n_feat = x.shape
        x = x.permute(0, 2, 1).reshape(batch * n_feat, 1, seq_len)
        summaries = []
        rate = torch.softmax((self.rate / self.temperature).squeeze(-1), dim=0)
        rate = rate * rate.numel()
        rate_index = 0

        for chain, module_list in zip(self.chains, self.chain_modules):
            conv_map = {}
            tmp = x
            for window, block in zip(chain, module_list):
                tmp = block["conv"](tmp)
                conv_map[window] = tmp

            last_window = None
            upscaled = tmp
            for window, block in reversed(list(zip(chain, module_list))):
                if last_window is not None:
                    scale_factor = last_window / window
                    upscaled = nn.functional.interpolate(upscaled, scale_factor=scale_factor)
                    diff = upscaled.shape[-1] - conv_map[window].shape[-1]
                    if diff < 0:
                        upscaled = nn.functional.pad(upscaled, (0, -diff), mode="replicate")
                    elif diff > 0:
                        upscaled = upscaled[:, :, :-diff]
                    rnn_input = upscaled + conv_map[window]
                else:
                    rnn_input = upscaled
                hidden, _ = block["gru"](rnn_input.permute(0, 2, 1))
                summary = hidden[:, -1, :].reshape(batch, n_feat, -1)
                summaries.append(summary * rate[rate_index])
                rate_index += 1
                last_window = window

        tokens = torch.cat(summaries, dim=-1)
        return self.dropout(self.out(tokens))


class AECCM(nn.Module):
    """re-mLANet4 AECCM: dense context update + low-rank update + fusion gate."""

    def __init__(self, config: DiffMLANetConfig):
        super().__init__()
        if config.d_model % config.aeccm_inplanes != 0:
            raise ValueError("d_model must be divisible by aeccm_inplanes")
        self.d_model = config.d_model
        self.inplanes = config.aeccm_inplanes
        planes = max(1, int(self.inplanes * config.aeccm_ratio))
        rank = max(1, int(self.inplanes * config.aeccm_low_rank_ratio))
        self.conv_mask = nn.Conv2d(self.inplanes, 1, kernel_size=1)
        self.softmax = nn.Softmax(dim=2)
        self.channel_add_conv1 = nn.Sequential(
            nn.Conv2d(self.inplanes, planes, kernel_size=1),
            nn.LayerNorm([planes, 1, 1]),
            nn.ReLU(inplace=True),
            nn.Conv2d(planes, self.inplanes, kernel_size=1),
        )
        self.channel_add_conv2 = nn.Sequential(
            nn.Conv2d(self.inplanes, planes, kernel_size=3, padding=1),
            nn.LayerNorm([planes, 1, 1]),
            nn.ReLU(inplace=True),
            nn.Conv2d(planes, self.inplanes, kernel_size=3, padding=1),
        )
        self.low_rank_down = nn.Conv2d(self.inplanes, rank, kernel_size=1, bias=False)
        self.low_rank_norm = nn.LayerNorm([rank, 1, 1])
        self.low_rank_mid = nn.Sequential(
            nn.Conv2d(rank, rank, kernel_size=3, padding=1, groups=rank, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(rank, rank, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
        )
        self.low_rank_up = nn.Conv2d(rank, self.inplanes, kernel_size=1, bias=False)
        self.fusion_gate = nn.Sequential(nn.Conv2d(self.inplanes * 2, self.inplanes, kernel_size=1), nn.Sigmoid())
        self.conv1 = nn.Conv2d(self.inplanes, self.inplanes, kernel_size=1)

    def spatial_attention(self, x: torch.Tensor) -> torch.Tensor:
        batch, channel, tokens, width = x.size()
        input_x = x.view(batch, channel, tokens * width).unsqueeze(1)
        context_mask = self.conv_mask(x).view(batch, 1, tokens * width)
        context_mask = self.softmax(context_mask).unsqueeze(-1)
        context = torch.matmul(input_x, context_mask)
        return context.view(batch, channel, 1, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, tokens, _ = x.shape
        y = x.reshape(batch, self.inplanes, tokens, self.d_model // self.inplanes)
        spatial_out = self.spatial_attention(y)
        dense_out = self.conv1(self.channel_add_conv2(spatial_out) + self.channel_add_conv1(spatial_out))
        low_rank = self.low_rank_up(self.low_rank_mid(self.low_rank_norm(self.low_rank_down(spatial_out))))
        gate = self.fusion_gate(torch.cat([dense_out, low_rank], dim=1))
        out = y + dense_out + gate * low_rank
        return out.reshape(batch, tokens, self.d_model)


class DiffTokenMixer(nn.Module):
    """Self-contained gated recurrent mixer used instead of external mLSTM files."""

    def __init__(self, config: DiffMLANetConfig):
        super().__init__()
        self.norm = nn.LayerNorm(config.d_model)
        self.gru = nn.GRU(
            input_size=config.d_model,
            hidden_size=config.d_model // 2,
            batch_first=True,
            bidirectional=True,
        )
        self.gate = nn.Sequential(nn.Linear(config.d_model, config.d_model), nn.Sigmoid())
        self.ffn = nn.Sequential(
            nn.Linear(config.d_model, config.d_model * 2),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.d_model * 2, config.d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mixed, _ = self.gru(self.norm(x))
        mixed = mixed * self.gate(x)
        return x + mixed + self.ffn(mixed)


class DiffMLANetFactorModel(nn.Module):
    def __init__(self, config: DiffMLANetConfig):
        super().__init__()
        self.config = config
        self.revin = RevIN(config.n_feat)
        self.diffusion_enabled = bool(config.diffusion_enabled)
        self.diffusion_denoise_eta = float(config.diffusion_denoise_eta)
        self.diffusion_denoiser = None
        if self.diffusion_enabled:
            self.diffusion_denoiser = DiffusionDenoiser(
                seq_len=config.seq_len,
                enc_in=config.n_feat,
                num_steps=config.diffusion_steps,
                beta_start=config.diffusion_beta_start,
                beta_end=config.diffusion_beta_end,
                hidden_dim=config.diffusion_hidden_dim,
                time_dim=config.diffusion_time_dim,
                num_blocks=config.diffusion_num_blocks,
                inference_step=config.diffusion_inference_step,
            )
        self.embedding = PyramidalRNNEmbedding(config)
        self.aeccm = AECCM(config)
        self.mixer1 = DiffTokenMixer(config)
        self.mixer2 = DiffTokenMixer(config)
        self.norm = nn.LayerNorm(config.d_model)
        self.projector = nn.Linear(config.d_model, 1)
        self.factor_head = nn.Sequential(
            nn.LayerNorm(config.n_feat * 3),
            nn.Linear(config.n_feat * 3, config.n_feat * 2),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.n_feat * 2, 1),
        )

    def _apply_diffusion_denoise(self, x: torch.Tensor) -> torch.Tensor:
        if self.diffusion_denoiser is None:
            return x
        x_clean = self.diffusion_denoiser(x)
        eta = max(0.0, min(float(self.diffusion_denoise_eta), 1.0))
        return x + eta * (x_clean - x)

    def forward(
        self,
        x: torch.Tensor,
        return_diffusion_loss: bool = False,
    ) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:
        last_features = x[:, -1, :]
        normalized = self.revin(x)
        diffusion_loss = normalized.new_tensor(0.0)
        if self.diffusion_enabled and self.diffusion_denoiser is not None:
            if return_diffusion_loss:
                diffusion_loss = self.diffusion_denoiser.training_loss(normalized)["loss"]
            normalized = self._apply_diffusion_denoise(normalized)
        tokens = self.embedding(normalized)
        tokens = self.aeccm(tokens)
        tokens = self.mixer2(self.mixer1(tokens))
        predicted_features = self.projector(self.norm(tokens)).squeeze(-1)
        diff_features = predicted_features - last_features
        factor_input = torch.cat([predicted_features, last_features, diff_features], dim=-1)
        score = self.factor_head(factor_input).squeeze(-1)
        if return_diffusion_loss:
            return score, diffusion_loss
        return score


def model_config() -> DiffMLANetConfig:
    return DiffMLANetConfig(
        n_feat=len(FEATURE_COLS),
        seq_len=SEQ_LEN,
        d_model=int(os.environ.get("DIFFMLANET_D_MODEL", "128")),
        conv_windows=[
            int(item)
            for item in os.environ.get("DIFFMLANET_CONV_WINDOWS", "4,8,16,32,64").split(",")
            if item.strip()
        ],
        conv_channels=int(os.environ.get("DIFFMLANET_CONV_CHANNELS", "96")),
        dropout=float(os.environ.get("DIFFMLANET_DROPOUT", "0.2")),
        diffusion_enabled=os.environ.get("DIFFMLANET_DIFFUSION_ENABLED", "1") != "0",
        diffusion_steps=int(os.environ.get("DIFFMLANET_DIFFUSION_STEPS", "128")),
        diffusion_beta_start=float(os.environ.get("DIFFMLANET_DIFFUSION_BETA_START", "0.0001")),
        diffusion_beta_end=float(os.environ.get("DIFFMLANET_DIFFUSION_BETA_END", "0.02")),
        diffusion_hidden_dim=int(os.environ.get("DIFFMLANET_DIFFUSION_HIDDEN_DIM", "64")),
        diffusion_time_dim=int(os.environ.get("DIFFMLANET_DIFFUSION_TIME_DIM", "64")),
        diffusion_num_blocks=int(os.environ.get("DIFFMLANET_DIFFUSION_NUM_BLOCKS", "2")),
        diffusion_inference_step=int(os.environ.get("DIFFMLANET_DIFFUSION_INFERENCE_STEP", "64")),
        diffusion_denoise_eta=float(os.environ.get("DIFFMLANET_DIFFUSION_DENOISE_ETA", "0.2")),
        lambda_diff=float(os.environ.get("DIFFMLANET_LAMBDA_DIFF", "0.01")),
    )


def build_model(config: DiffMLANetConfig | dict[str, Any] | None = None) -> DiffMLANetFactorModel:
    if isinstance(config, dict):
        base = DiffMLANetConfig()
        for key, value in config.items():
            if hasattr(base, key):
                setattr(base, key, value)
        config = base
    elif config is None:
        config = model_config()
    return DiffMLANetFactorModel(config)

def load_fusion_json(model_path: str | os.PathLike = MODEL_PATH) -> dict[str, Any]:
    payload = json.loads(Path(model_path).read_text(encoding="utf-8"))
    if payload.get("fusion_type") != "daily_rank_weighted_sum":
        raise RuntimeError(f"不是支持的融合模型: {payload.get('fusion_type')}")
    if "models" not in payload or "1m" not in payload["models"] or "5m" not in payload["models"]:
        raise RuntimeError("融合 model.json 必须包含 models['1m'] 和 models['5m']")
    weights = payload.get("weights", {})
    if abs(float(weights.get("1m", 0.0)) + float(weights.get("5m", 0.0)) - 1.0) > 1e-8:
        raise RuntimeError(f"融合权重之和必须为 1: {weights}")
    return payload


def load_submodel_from_payload(payload: dict[str, Any], *, freq: str, device: torch.device):
    if payload.get("feature_cols") != list(FEATURE_COLS):
        raise RuntimeError(f"{freq} 子模型字段列表与当前 infer.py 不一致")
    expected_schema = SCHEMA_BY_FREQ[freq]
    if payload.get("feature_schema_version") != expected_schema:
        raise RuntimeError(
            f"{freq} 子模型字段版本不一致: {payload.get('feature_schema_version')} != {expected_schema}"
        )
    if payload.get("metadata", {}).get("smoke_test") and os.environ.get("DIFFMLANET_ALLOW_SMOKE_WEIGHT") != "1":
        raise RuntimeError(f"{freq} 子模型是小样本权重，禁止正式提交")

    model = build_model(payload["model_cfg"]).to(device)
    state = {key: torch.tensor(value, dtype=torch.float32) for key, value in payload["state_dict"].items()}
    model.load_state_dict(state)
    model.eval()
    stats = payload["stats"]
    mean = np.asarray(stats["mean"], dtype=np.float32)
    std = np.asarray(stats["std"], dtype=np.float32)
    seq_len = int(payload.get("model_cfg", {}).get("seq_len", SEQ_LEN))
    return model, (mean, std), seq_len


def _predict_batches(model: torch.nn.Module, X: np.ndarray, *, device: torch.device) -> np.ndarray:
    output = []
    tensor = torch.from_numpy(X)
    with torch.inference_mode():
        for start in range(0, len(tensor), BATCH):
            batch = tensor[start:start + BATCH].to(device, non_blocking=True)
            output.append(model(batch).detach().cpu().numpy())
    return np.concatenate(output).astype(np.float64)


def _require_dai() -> None:
    if dai is None:
        raise ModuleNotFoundError("云端推理需要 BigQuant dai 环境")


def _constituents(start_date: str, end_date: str) -> pd.DataFrame:
    _require_dai()
    frame = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
    frame["instrument"] = frame["instrument"].astype(str)
    return frame.drop_duplicates(["date", "instrument"])



def _month_ranges(start_date: str, end_date: str) -> list[tuple[str, str]]:
    """把平台传入的大区间拆成按月小区间，降低单次 dai.query 内存。"""
    cur = pd.Timestamp(start_date).normalize()
    end = pd.Timestamp(end_date).normalize()
    ranges: list[tuple[str, str]] = []
    while cur <= end:
        month_end = cur + pd.offsets.MonthEnd(0)
        sub_end = min(month_end, end)
        ranges.append(
            (
                f"{cur:%Y-%m-%d} 00:00:00",
                f"{sub_end:%Y-%m-%d} 23:59:59",
            )
        )
        cur = sub_end + pd.Timedelta(days=1)
    return ranges


def _postprocess_daily_scores(result: pd.DataFrame) -> pd.DataFrame:
    """输出前做每日截面后处理，使 score 更贴近比赛的截面排序目标。"""
    result = result.copy()
    result["score"] = (
        pd.to_numeric(result["score"], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .astype(float)
    )
    if SCORE_POSTPROCESS in {"daily_rank", "rank", "rank_pct"}:
        result["score"] = (
            result.groupby("date")["score"]
            .rank(method="average", pct=True)
            .sub(0.5)
            .mul(2.0)
            .astype(float)
        )
    elif SCORE_POSTPROCESS in {"none", "raw"}:
        pass
    else:
        raise ValueError(f"未知 DIFFMLANET_SCORE_POSTPROCESS={SCORE_POSTPROCESS!r}")
    return result


def _daily_rank(series: pd.Series) -> pd.Series:
    return series.rank(method="average", pct=True).sub(0.5).mul(2.0).astype(float)


def _query_model_scores(
    *,
    freq: str,
    submodel_payload: dict[str, Any],
    datasources: dict,
    start_date: str,
    end_date: str,
    constituents: pd.DataFrame,
) -> pd.DataFrame:
    """对单个频率子模型推理，返回未 rank 的 raw score，并保留全部 constituents。"""
    global SEQ_LEN

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, stats, seq_len = load_submodel_from_payload(submodel_payload, freq=freq, device=device)
    old_seq_len = SEQ_LEN
    SEQ_LEN = seq_len

    try:
        instruments = sorted(constituents["instrument"].unique().tolist())
        table = datasources.get(f"bar{freq}") or datasources.get(freq) or TABLE_BY_FREQ[freq]
        sql = f"SELECT date, instrument, {', '.join(FEATURE_COLS)} FROM {table} ORDER BY instrument, date"
        parts: list[pd.DataFrame] = []

        for sub_start, sub_end in _month_ranges(start_date, end_date):
            bar_start, bar_end = full_day_bounds(sub_start, sub_end)
            log("推理月份", freq=freq, table=table, start=bar_start, end=bar_end, instruments=len(instruments))
            for offset in range(0, len(instruments), INFER_CHUNK):
                chunk = instruments[offset:offset + INFER_CHUNK]
                raw = dai.query(sql, filters={"date": [bar_start, bar_end], "instrument": chunk}).df()
                if raw.empty:
                    continue
                canonical = to_canonical(raw, is_local=False)
                try:
                    X, _, index_df, _ = build_samples(
                        canonical,
                        start_date=bar_start,
                        end_date=bar_end,
                        mode="infer",
                        instruments=chunk,
                        stats=stats,
                    )
                except RuntimeError as exc:
                    log(
                        "推理分块跳过",
                        freq=freq,
                        instruments=f"{min(offset + len(chunk), len(instruments))}/{len(instruments)}",
                        reason=str(exc),
                    )
                    continue
                index_df["score"] = _predict_batches(model, X, device=device)
                parts.append(index_df)
                log("云端推理", freq=freq, instruments=f"{min(offset + len(chunk), len(instruments))}/{len(instruments)}", rows=len(index_df))

        if parts:
            scores = pd.concat(parts, ignore_index=True)
            scores["date"] = pd.to_datetime(scores["date"]).dt.normalize()
            scores["instrument"] = scores["instrument"].astype(str)
            scores = (
                scores.replace([np.inf, -np.inf], np.nan)
                .dropna(subset=["score"])
                .drop_duplicates(["date", "instrument"], keep="last")
            )
        else:
            scores = pd.DataFrame(columns=["date", "instrument", "score"])

        raw_expected = constituents.groupby("date").size()
        raw_actual = scores.groupby("date").size().reindex(raw_expected.index, fill_value=0)
        raw_coverage = raw_actual / raw_expected
        if len(raw_coverage) > 0 and float(raw_coverage.min()) < MIN_DAILY_COVERAGE:
            log(
                "原始预测覆盖率偏低，已用 0 填充缺失分数",
                freq=freq,
                min_coverage=round(float(raw_coverage.min()), 4),
                threshold=MIN_DAILY_COVERAGE,
            )

        result = constituents.merge(scores, on=["date", "instrument"], how="left")
        result["score"] = (
            pd.to_numeric(result["score"], errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
            .astype(float)
        )
        return result[["date", "instrument", "score"]]
    finally:
        SEQ_LEN = old_seq_len
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def main(datasources: dict, start_date: str, end_date: str) -> pd.DataFrame:
    fusion = load_fusion_json(MODEL_PATH)
    constituents = _constituents(start_date, end_date)
    if constituents.empty:
        raise RuntimeError(f"平台传入区间没有中证1000成分股: {start_date} ~ {end_date}")

    weights = fusion["weights"]
    w1 = float(weights["1m"])
    w5 = float(weights["5m"])
    log("开始融合推理", weight_1m=w1, weight_5m=w5, rows=len(constituents))

    score_1m = _query_model_scores(
        freq="1m",
        submodel_payload=fusion["models"]["1m"],
        datasources=datasources,
        start_date=start_date,
        end_date=end_date,
        constituents=constituents,
    ).rename(columns={"score": "score_1m"})

    score_5m = _query_model_scores(
        freq="5m",
        submodel_payload=fusion["models"]["5m"],
        datasources=datasources,
        start_date=start_date,
        end_date=end_date,
        constituents=constituents,
    ).rename(columns={"score": "score_5m"})

    merged = constituents.merge(score_1m, on=["date", "instrument"], how="left")
    merged = merged.merge(score_5m, on=["date", "instrument"], how="left")
    merged["score_1m"] = pd.to_numeric(merged["score_1m"], errors="coerce").fillna(0.0)
    merged["score_5m"] = pd.to_numeric(merged["score_5m"], errors="coerce").fillna(0.0)

    merged["rank_1m"] = merged.groupby("date")["score_1m"].transform(_daily_rank)
    merged["rank_5m"] = merged.groupby("date")["score_5m"].transform(_daily_rank)
    merged["score"] = w1 * merged["rank_1m"] + w5 * merged["rank_5m"]

    result = (
        merged[["date", "instrument", "score"]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["score"])
        .drop_duplicates(["date", "instrument"], keep="last")
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
    )
    log("融合推理完成", rows=len(result), days=result["date"].nunique(), weight_1m=w1, weight_5m=w5)
    return result


def _platform_datasources() -> dict:
    g = globals()
    if "datasources" in g:
        value = g["datasources"]
        if isinstance(value, str):
            return {"bar1m": value}
        return dict(value)
    if "datasource" in g:
        value = g["datasource"]
        if isinstance(value, str):
            return {"bar1m": value}
        return dict(value)
    return {
        "bar1m": os.environ.get("FUSION_CLOUD_BAR1M_TABLE", "bigalpha_2026_stock_bar1m"),
        "bar5m": os.environ.get("FUSION_CLOUD_BAR5M_TABLE", "bigalpha_2026_stock_bar5m"),
    }


def _platform_dates() -> tuple[str, str]:
    g = globals()
    if "start_date" in g and "end_date" in g:
        return str(g["start_date"]), str(g["end_date"])
    return (
        os.environ.get("FUSION_START_DATE", "2024-01-01 00:00:00"),
        os.environ.get("FUSION_END_DATE", "2024-12-31 23:59:59"),
    )


if __name__ == "__main__":
    platform_datasources = _platform_datasources()
    start_value, end_value = _platform_dates()
    score_data = main(platform_datasources, start_value, end_value)
    print(score_data.head())

    try:
        from bigmodule import M

        result = M.bigalpha_eval._latest(factor_data=score_data, show=True)
        print(result)
    except ModuleNotFoundError:
        log("未检测到 bigmodule，跳过平台评估；本地/非评估环境可忽略")


[reDMFusion][infer] 开始融合推理 weight_1m=0.8 weight_5m=0.2 rows=242000
[reDMFusion][infer] 推理月份 freq=1m table=bigalpha_2026_stock_bar1m start=2024-01-01 00:00:00 end=2024-01-31 23:59:59 instruments=1206
[reDMFusion][infer] 云端推理 freq=1m instruments=80/1206 rows=1751
[reDMFusion][infer] 云端推理 freq=1m instruments=160/1206 rows=1738
[reDMFusion][infer] 云端推理 freq=1m instruments=240/1206 rows=1760


KeyboardInterrupt: 